# VisoSwap — Free GPU Runner on Kaggle

Run VisoSwap Studio on Kaggle's free GPU (NVIDIA T4 or P100).

### Instructions:
1. In the right panel under **Settings**:
   - **Accelerator**: Select **GPU T4 x2** (or **GPU P100**)
   - **Internet**: Toggle to **Internet on** (required for downloads and Cloudflare tunnel)
2. Click **Run All** (or run cells one-by-one).
3. Wait for the public Cloudflare tunnel URL printed at the bottom cell (e.g. `https://xxxx.trycloudflare.com`).
4. Open the link on any browser or mobile device (Safari/Chrome) to use VisoSwap with live stream swap and theater mode!

In [ ]:
# Step 1: Verify GPU Environment
!nvidia-smi

In [ ]:
# Step 2: Install System Dependencies & Cloudflare Tunnel
!apt-get update -qq && apt-get install -y -qq ffmpeg curl
!curl -fsSL https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -o /usr/local/bin/cloudflared
!chmod +x /usr/local/bin/cloudflared
!cloudflared --version

In [ ]:
# Step 3: Clone VisoSwap Repository (including pre-built Web UI)
import os
if not os.path.exists('/kaggle/working/visoswap'):
    !git clone --depth 1 https://github.com/kaiser62/visoswap.git /kaggle/working/visoswap
%cd /kaggle/working/visoswap
!ls -la frontend/dist

In [ ]:
# Step 4: Install Python & Engine Dependencies with CUDA support
!pip install --no-cache-dir \
    fastapi uvicorn[standard] pydantic pydantic-settings aiosqlite httpx \
    python-multipart yt-dlp opencv-python Pillow ftfy regex numexpr onnxsim requests tqdm

# Ensure onnxruntime-gpu matches Kaggle CUDA runtime
!pip install --no-cache-dir onnxruntime-gpu

In [ ]:
# Step 5: Download & Bootstrap All Required Models (~12GB)
# Downloads directly to model_assets_owned/ with integrity check
import os
os.environ['MODELS_DIR'] = '/kaggle/working/visoswap/model_assets_owned'
os.makedirs('/kaggle/working/visoswap/model_assets_owned', exist_ok=True)

from visoswap.models.bootstrap import repair, verify, FAST, FULL

print('Starting model download and verification...')
res = repair(models_dir='/kaggle/working/visoswap/model_assets_owned', mode=FAST)
print('Bootstrap status: Complete!')
print(f'Required models checked: {res.required_checked}, Present: {len(res.present)}')
if not res.ok:
    print('Missing entries:', [e.name for e in res.absent])
    print('Mismatching entries:', [e.name for e in res.mismatching])
    raise RuntimeError('Model download incomplete!')

In [ ]:
# Step 6: Launch Cloudflare Tunnel and Start VisoSwap Backend Server
import subprocess
import time
import re

# 1. Start Cloudflare Tunnel in background
tunnel_log = open('/kaggle/working/tunnel.log', 'w')
tunnel_proc = subprocess.Popen(
    ['cloudflared', 'tunnel', '--url', 'http://127.0.0.1:8000', '--no-autoupdate'],
    stdout=tunnel_log,
    stderr=subprocess.STDOUT
)

# 2. Extract public trycloudflare.com URL
print('Waiting for Cloudflare Tunnel to connect...')
public_url = None
for _ in range(30):
    time.sleep(1)
    if os.path.exists('/kaggle/working/tunnel.log'):
        content = open('/kaggle/working/tunnel.log').read()
        match = re.search(r'https://[a-zA-Z0-9-]+\.trycloudflare\.com', content)
        if match:
            public_url = match.group(0)
            break

if public_url:
    print('=' * 65)
    print('🚀 VisoSwap is LIVE! Access your Web UI at:')
    print(f'👉 {public_url}')
    print(f'👉 Mobile UI: {public_url}/mobile')
    print('=' * 65)
else:
    print('Could not find tunnel URL yet. Check /kaggle/working/tunnel.log')

# 3. Run FastAPI/Uvicorn server
!python -m uvicorn backend.main:app --host 0.0.0.0 --port 8000
